# Quantum Machine Learning for Option Pricing — Baseline Experiment (CPU)

## Overview

This notebook implements and compares two training paradigms — **supervised** and **unsupervised** — for pricing European put options using a **Parameterised Quantum Circuit (PQC)** as a probability distribution learner. The model is trained on simulated Black-Scholes data and evaluated against analytical Black-Scholes prices.

The experiment sweeps over three strike prices (OTM, ATM, ITM) and multiple dataset sizes to study how each training mode scales with data.

---

## Problem Setup

We model the **log-moneyness** $x_t = \log(S_T / K)$ of the underlying asset at maturity as a random variable. The PQC learns its distribution (CDF/PDF). Once trained, the risk-neutral option price is recovered via Fourier integration.

**Black-Scholes parameters:**
- Initial stock price $S_0 = 100$, risk-free rate $r = 10\%$, volatility $\sigma = 25\%$, maturity $T = 1$ year
- Strike prices $K \in \{90, 100, 110\}$ (in-the-money, at-the-money, out-of-the-money)

**Data pipeline:**
1. Sample $S_T \sim $ Log-Normal$(\log S_0 + (r - \sigma^2/2)T,\ \sigma^2 T)$
2. Compute log-moneyness $x_t = \log(S_T / K)$
3. Rescale $x_t$ to $[-\pi, \pi]$ for PQC encoding

---

## Quantum Circuit Architecture

The PQC is a **hardware-efficient ansatz** built with `qml4var.architectures.hardware_efficient_ansatz`:
- **Supervised mode:** 3 qubits/feature × 3 layers
- **Unsupervised mode:** 2 qubits/feature × 2 layers
- Single input feature (log-moneyness), encoded in $[-\pi, \pi]$
- The circuit output is interpreted as a **CDF** in $[-1, 1]$; its derivative gives the **PDF**
- Trained with **PyTorch autograd** (GPU-compatible backpropagation)

---

## Training Modes

### Supervised
- **Target:** Empirical CDF of the training data, shifted to $[-0.5, 0.5]$
- **Loss:** Weighted sum of MSE (weight 0.9) + regularisation term (weight 0.1)
- **Learning rate:** 0.005 | **Dataset sizes:** 250, 500, 1000 samples

The model fits the circuit output directly to the empirical CDF. This requires labelled data (sample points paired with CDF values), making it analogous to standard supervised regression.

### Unsupervised
- **Target:** No explicit labels — the model learns from the data distribution alone
- **Loss:** `unsupervised_qdml_loss_workflow` encourages the PDF (circuit derivative) to match the empirical data density. Weights: [0.2 PDF loss, 0.8 auxiliary loss]
- **Learning rate:** 0.1 | **Dataset sizes:** 1000, 2500, 5000 samples

The model learns by minimising a distributional divergence, without needing paired (x, CDF(x)) labels. More data is needed since the signal is weaker.

**Common optimizer settings:** Adam with $\beta_1=0.9$, $\beta_2=0.999$, early stopping (patience = 25 epochs, tolerance = $10^{-8}$), max 50 epochs.

---

## Option Pricing Methods

Once the circuit is trained, the put option price $V = e^{-rT} \mathbb{E}[\max(K - S_T, 0)]$ is estimated via **Fourier series integration** on a 1024-point grid over $[-2\pi, 2\pi]$.

### Method I — PDF-based Fourier pricing
1. Evaluate the circuit PDF on a dense grid and normalise it to unit area
2. Compute Fourier coefficients $(A_k, B_k)$ of the PDF ($K=12$ terms)
3. Compute Fourier coefficients of the put payoff $h(x) = \max(K(1 - e^x), 0)$
4. Apply the Fourier pricing formula via `finance.fourier_price_v_t0`

*Used by: supervised mode*

### Method II — Integration by Parts (IBP)
1. Treat the circuit output directly as a CDF; extend to $[-4\pi, 4\pi]$ with boundary conditions $F(-4\pi)=0$, $F(4\pi)=1$
2. Compute CDF Fourier coefficients on the extended interval
3. Locate the discontinuity point $c$ where $x_{\text{raw}} = 0$ (i.e. $S_T = K$)
4. Compute the payoff derivative $h'(u)$ analytically and split the integral at $c$
5. Apply the IBP formula: $V = e^{-rT}\bigl[h(b)F(b) - h(a)F(a) - \int_a^b h'(u)F(u)\,du\bigr]$

IBP avoids differentiating the learned PDF, leading to lower variance estimates. *Used by: unsupervised mode*

---

## Evaluation

For each (mode, strike, dataset size) combination:
- **Test MSE:** Mean squared error of the circuit CDF predictions on held-out test points
- **Estimated price vs. Black-Scholes price:** Absolute and relative error
- **Runtime:** Seconds per training run

Results are aggregated (mean ± std over repetitions) and plotted as **price curves vs. dataset size** for each strike, with the theoretical Black-Scholes price as a dashed reference line.

---

## Execution

- **Device:** CUDA GPU if available, otherwise CPU
- **Parallelism:** None — runs sequentially (set `DASK_CLIENT = None`)
- **Debug mode:** Set `DEBUG_FAST_RUN = True` to run 2 epochs with the smallest dataset for a quick sanity check

In [ ]:
import sys
from time import perf_counter

import torch

sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats

from finance import (
    bs_put_price,
    estimate_price_from_trained_pqc,
    estimate_price_ibp,
)
from qml4var.adam import adam_optimizer_loop
from qml4var.architectures import (
    hardware_efficient_ansatz,
    init_weights,
    normalize_data,
)
from qml4var.data_utils import (
    empirical_cdf,
    inverse_rescaling_u_to_xt,
    simulate_black_scholes_data_rescaled,
)
from qml4var.losses import torch_gradient
from qml4var.workflows import (
    qdml_loss_workflow,
    unsupervised_qdml_loss_workflow,
    workflow_for_cdf,
    workflow_for_pdf,
)

try:
    from tqdm.auto import tqdm

    TQDM_AVAILABLE = True
except Exception:
    TQDM_AVAILABLE = False

    class _NoTqdm:
        def __init__(self, total=None, desc=None, unit=None, disable=False):
            self.total = total
            self.n = 0
            self.disable = disable
            if not self.disable:
                print(f"{desc or 'Progress'} | total={total} {unit or ''}")

        def update(self, n=1):
            self.n += n

        def set_postfix(self, *args, **kwargs):
            return None

        def close(self):
            return None

    def tqdm(*args, **kwargs):
        return _NoTqdm(*args, **kwargs)


## 1. Global Setup

In [ ]:
# Black-Scholes parameters
S0 = 100.0
r = 0.10
sigma = 0.25
T = 1.0
STRIKES = [90, 100, 110]

# Domains
TRAIN_INTERVAL = (-2.0 * np.pi, 2.0 * np.pi)  # workflow domain
DATA_INTERVAL = (-1.0 * np.pi, 1.0 * np.pi)  # data interval after rescaling
ENCODING_INTERVAL = (-1.0 * np.pi, 1.0 * np.pi)
FEATURES_NUMBER = 1

# Fourier pricing setup
K_TERMS = 12
GRID_POINTS_FOURIER = 1024

# Optional: use Dask client
DASK_CLIENT = None

# Torch device (GPU if available, else CPU)
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("TORCH_DEVICE:", TORCH_DEVICE)

# Normalization from training domain [-2pi, 2pi] into encoding [-pi, pi]
base_frecuency, shift_feature = normalize_data(
    [TRAIN_INTERVAL[0]] * FEATURES_NUMBER,
    [TRAIN_INTERVAL[1]] * FEATURES_NUMBER,
    [ENCODING_INTERVAL[0]] * FEATURES_NUMBER,
    [ENCODING_INTERVAL[1]] * FEATURES_NUMBER,
)

print("base_frecuency:", base_frecuency)
print("shift_feature:", shift_feature)


## 2. Independent Configurations by Mode

You can edit each mode independently, without touching training functions.


In [3]:
MODE_CONFIGS = {
    "supervised": {
        "learning_rate": 0.005,
        "epochs": 50,
        "loss_weights": [0.9, 0.1],
        "train_sizes": [250, 500, 1000],
        "test_points": 100,
        "repetitions": 1,
        "beta1": 0.9,
        "beta2": 0.999,
        "tolerance": 1.0e-8,
        "n_counts_tolerance": 25,
        "print_step": 50,
        "batch_size": None,
        "n_qubits_by_feature": 3,
        "n_layers": 3,
        "debug_price_stats": True,
        "show_epoch_progress": True,
        "epoch_progress_leave": False,
    },
    "unsupervised": {
        "learning_rate": 0.1,
        "epochs": 50,
        "loss_weights": [0.2, 0.8],
        "train_sizes": [1000, 2500, 5000],
        "test_points": 1000,
        "repetitions": 1,
        "beta1": 0.9,
        "beta2": 0.999,
        "tolerance": 1.0e-8,
        "n_counts_tolerance": 25,
        "print_step": 50,
        "batch_size": None,
        "empirical_shift": -0.5,
        "n_qubits_by_feature": 2,
        "n_layers": 2,
        "debug_price_stats": True,
        "show_epoch_progress": True,
        "epoch_progress_leave": False,
    },
}

MODE_CONFIGS

{'supervised': {'learning_rate': 0.005,
  'epochs': 50,
  'loss_weights': [0.9, 0.1],
  'train_sizes': [250, 500, 1000],
  'test_points': 100,
  'repetitions': 1,
  'beta1': 0.9,
  'beta2': 0.999,
  'tolerance': 1e-08,
  'n_counts_tolerance': 25,
  'print_step': 50,
  'batch_size': None,
  'n_qubits_by_feature': 3,
  'n_layers': 3,
  'debug_price_stats': True,
  'show_epoch_progress': True,
  'epoch_progress_leave': False},
 'unsupervised': {'learning_rate': 0.1,
  'epochs': 50,
  'loss_weights': [0.2, 0.8],
  'train_sizes': [1000, 2500, 5000],
  'test_points': 1000,
  'repetitions': 1,
  'beta1': 0.9,
  'beta2': 0.999,
  'tolerance': 1e-08,
  'n_counts_tolerance': 25,
  'print_step': 50,
  'batch_size': None,
  'empirical_shift': -0.5,
  'n_qubits_by_feature': 2,
  'n_layers': 2,
  'debug_price_stats': True,
  'show_epoch_progress': True,
  'epoch_progress_leave': False}}

## 3. Helper Functions

In [ ]:
def build_mode_artifacts(cfg):
    pqc_cfg = {
        "features_number": FEATURES_NUMBER,
        "n_qubits_by_feature": int(cfg["n_qubits_by_feature"]),
        "n_layers": int(cfg["n_layers"]),
        "base_frecuency": list(base_frecuency),
        "shift_feature": list(shift_feature),
    }
    circuit_fn, weights_names, features_names = hardware_efficient_ansatz(**pqc_cfg)

    workflow_cfg = {
        "circuit_fn": circuit_fn,
        "weights_names": weights_names,
        "features_names": features_names,
        "torch_device": TORCH_DEVICE,
        "minval": [TRAIN_INTERVAL[0]],
        "maxval": [TRAIN_INTERVAL[1]],
        "points": 150,
    }
    return {
        "pqc_cfg": pqc_cfg,
        "weights_names": weights_names,
        "workflow_cfg": workflow_cfg,
    }


def batch_generator(X, Y, batch_size):
    return [(X[i : i + batch_size], Y[i : i + batch_size]) for i in range(0, len(X), batch_size)]


def run_supervised(u_train, cfg, artifacts, progress_desc=None):
    workflow_cfg = artifacts["workflow_cfg"]
    weights_names = artifacts["weights_names"]

    y_train = empirical_cdf(u_train).reshape((-1, 1)) - 0.5
    batch_size = cfg["batch_size"] if cfg["batch_size"] is not None else len(u_train)
    batches = batch_generator(u_train, y_train, batch_size)

    def training_loss(w_):
        return qdml_loss_workflow(
            w_, u_train, y_train, dask_client=DASK_CLIENT, loss_weights=cfg["loss_weights"], **workflow_cfg
        )

    def loss_for_grad(w_, x_, y_):
        return qdml_loss_workflow(
            w_, x_, y_, dask_client=DASK_CLIENT, loss_weights=cfg["loss_weights"], **workflow_cfg
        )

    def gradient_fn(w_, x_, y_):
        return torch_gradient(w_, x_, y_, loss_for_grad)

    optimizer_cfg = {
        "epochs": cfg["epochs"],
        "learning_rate": cfg["learning_rate"],
        "beta1": cfg["beta1"],
        "beta2": cfg["beta2"],
        "tolerance": cfg["tolerance"],
        "n_counts_tolerance": cfg["n_counts_tolerance"],
        "print_step": cfg["print_step"],
        "progress_bar": bool(cfg.get("show_epoch_progress", False)),
        "progress_desc": progress_desc if progress_desc is not None else "supervised epochs",
        "progress_leave": bool(cfg.get("epoch_progress_leave", False)),
    }

    initial_weights = init_weights(weights_names)
    return adam_optimizer_loop(
        weights_dict=initial_weights,
        loss_function=training_loss,
        metric_function=None,
        gradient_function=gradient_fn,
        batch_generator=batches,
        initial_time=0,
        **optimizer_cfg,
    )


def run_unsupervised(u_train, cfg, artifacts, progress_desc=None):
    workflow_cfg = artifacts["workflow_cfg"]
    weights_names = artifacts["weights_names"]

    dummy_y = np.zeros((u_train.shape[0], 1))
    batch_size = cfg["batch_size"] if cfg["batch_size"] is not None else len(u_train)
    batches = batch_generator(u_train, dummy_y, batch_size)

    def training_loss(w_):
        return unsupervised_qdml_loss_workflow(
            w_,
            u_train,
            dask_client=DASK_CLIENT,
            empirical_shift=cfg["empirical_shift"],
            loss_weights=cfg["loss_weights"],
            **workflow_cfg,
        )

    def loss_for_grad(w_, x_, y_):
        return unsupervised_qdml_loss_workflow(
            w_,
            x_,
            dask_client=DASK_CLIENT,
            empirical_shift=cfg["empirical_shift"],
            loss_weights=cfg["loss_weights"],
            **workflow_cfg,
        )

    def gradient_fn(w_, x_, y_):
        return torch_gradient(w_, x_, y_, loss_for_grad)

    optimizer_cfg = {
        "epochs": cfg["epochs"],
        "learning_rate": cfg["learning_rate"],
        "beta1": cfg["beta1"],
        "beta2": cfg["beta2"],
        "tolerance": cfg["tolerance"],
        "n_counts_tolerance": cfg["n_counts_tolerance"],
        "print_step": cfg["print_step"],
        "progress_bar": bool(cfg.get("show_epoch_progress", False)),
        "progress_desc": progress_desc if progress_desc is not None else "unsupervised epochs",
        "progress_leave": bool(cfg.get("epoch_progress_leave", False)),
    }

    initial_weights = init_weights(weights_names)
    return adam_optimizer_loop(
        weights_dict=initial_weights,
        loss_function=training_loss,
        metric_function=None,
        gradient_function=gradient_fn,
        batch_generator=batches,
        initial_time=0,
        **optimizer_cfg,
    )


## 4. Debug

Use these toggles before launching the full loop.


In [5]:
# Optional quick sanity mode
DEBUG_FAST_RUN = False

# More frequent progress logs (helps confirm the process is alive)
VERBOSE_PROGRESS = True

if VERBOSE_PROGRESS:
    for cfg in MODE_CONFIGS.values():
        cfg["print_step"] = min(int(cfg.get("print_step", 50)), 10)
        cfg["print_step"] = max(cfg["print_step"], 1)

if DEBUG_FAST_RUN:
    # tiny run to verify everything end-to-end before expensive execution
    for cfg in MODE_CONFIGS.values():
        cfg["epochs"] = min(int(cfg["epochs"]), 2)
        cfg["repetitions"] = 1
        cfg["train_sizes"] = [int(cfg["train_sizes"][0])]

MODE_CONFIGS


{'supervised': {'learning_rate': 0.005,
  'epochs': 50,
  'loss_weights': [0.9, 0.1],
  'train_sizes': [250, 500, 1000],
  'test_points': 100,
  'repetitions': 1,
  'beta1': 0.9,
  'beta2': 0.999,
  'tolerance': 1e-08,
  'n_counts_tolerance': 25,
  'print_step': 10,
  'batch_size': None,
  'n_qubits_by_feature': 3,
  'n_layers': 3,
  'debug_price_stats': True,
  'show_epoch_progress': True,
  'epoch_progress_leave': False},
 'unsupervised': {'learning_rate': 0.1,
  'epochs': 50,
  'loss_weights': [0.2, 0.8],
  'train_sizes': [1000, 2500, 5000],
  'test_points': 1000,
  'repetitions': 1,
  'beta1': 0.9,
  'beta2': 0.999,
  'tolerance': 1e-08,
  'n_counts_tolerance': 25,
  'print_step': 10,
  'batch_size': None,
  'empirical_shift': -0.5,
  'n_qubits_by_feature': 2,
  'n_layers': 2,
  'debug_price_stats': True,
  'show_epoch_progress': True,
  'epoch_progress_leave': False}}

In [6]:
def expected_total_runs(mode_configs, strikes):
    return sum(len(cfg["train_sizes"]) * len(strikes) * int(cfg["repetitions"]) for cfg in mode_configs.values())


def format_eta(seconds):
    if seconds is None or (not np.isfinite(seconds)):
        return "n/a"
    seconds = int(max(0, round(seconds)))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def eta_from_partial(results_df, mode_configs, strikes):
    total_runs = expected_total_runs(mode_configs, strikes)
    done_runs = len(results_df)
    if done_runs == 0:
        return {"total_runs": total_runs, "done_runs": 0, "avg_sec_per_run": None, "eta_hours": None}
    avg_sec = float(results_df["seconds"].mean())
    pending = max(total_runs - done_runs, 0)
    eta_hours = (pending * avg_sec) / 3600.0
    return {
        "total_runs": total_runs,
        "done_runs": done_runs,
        "avg_sec_per_run": avg_sec,
        "eta_hours": eta_hours,
    }


USE_TQDM = True

# Usage after (or during) the main loop:
# eta_from_partial(results_df, MODE_CONFIGS, STRIKES)


## 4. Run Experiment 

In [7]:
import os

os.environ["QML4VAR_PROFILE_TRAINING"] = "1"
os.environ["QML4VAR_PROFILE_EPOCHS"] = "1"
os.environ["QML4VAR_PROFILE_ONCE"] = "1"
os.environ["QML4VAR_PROFILE_LABEL"] = "first_case"

os.environ["QML4VAR_PROFILE_GRAD"] = "1"
os.environ["QML4VAR_PROFILE_GRAD_FIRST_ONLY"] = "1"


In [ ]:
results = []
base_seed = 2026

mode_artifacts = {}
for mode_name, cfg in MODE_CONFIGS.items():
    artifacts = build_mode_artifacts(cfg)
    mode_artifacts[mode_name] = artifacts
    print(
        f"mode={mode_name} -> qubits_by_feature={cfg['n_qubits_by_feature']} "
        f"layers={cfg['n_layers']} total_weights={len(artifacts['weights_names'])}"
    )

total_runs = expected_total_runs(MODE_CONFIGS, STRIKES)
print(f"\nPlanned training runs: {total_runs}")

progress = tqdm(
    total=total_runs,
    desc="Training runs",
    unit="run",
    disable=not (USE_TQDM and TQDM_AVAILABLE),
)

run_counter = 0
global_t0 = perf_counter()

for mode_name, cfg in MODE_CONFIGS.items():
    artifacts = mode_artifacts[mode_name]
    print(f"\n=== Running mode: {mode_name} ===")
    for K_ in STRIKES:
        theor_price = bs_put_price(S0, K_, r, sigma, T)
        for n_train in cfg["train_sizes"]:
            n_test = int(cfg["test_points"])
            for rep in range(cfg["repetitions"]):
                seed = base_seed + 100000 * rep + 1000 * int(K_) + n_train
                n_total = n_train + n_test

                x_raw_all, u_all, x_min_raw, x_max_raw = simulate_black_scholes_data_rescaled(
                    S0_=S0, r_=r, T_=T, sigma_=sigma, K_=K_, n_points=n_total, seed=seed
                )
                u_train = u_all[:n_train]
                u_test = u_all[n_train:]

                t0 = perf_counter()
                epoch_desc = f"{mode_name} K={K_} n={n_train} rep={rep + 1}"
                if mode_name == "supervised":
                    weights = run_supervised(u_train, cfg, artifacts, progress_desc=epoch_desc)
                else:
                    weights = run_unsupervised(u_train, cfg, artifacts, progress_desc=epoch_desc)
                t1 = perf_counter()

                # Optional test metric (CDF MSE wrt empirical CDF on test split)
                y_test = empirical_cdf(u_test).reshape((-1, 1)) - 0.5
                y_pred_test = workflow_for_cdf(weights, u_test, dask_client=DASK_CLIENT, **artifacts["workflow_cfg"])[
                    "y_predict_cdf"
                ].reshape((-1, 1))
                test_mse = float(np.mean((y_pred_test - y_test) ** 2))

                debug_label = f"mode={mode_name} K={K_} n_train={n_train} rep={rep + 1}"

                # ---- Pricing: Method I uses PDF-based formula; Method II uses IBP ----
                if mode_name == "supervised":
                    est_price = estimate_price_from_trained_pqc(
                        weights=weights,
                        artifacts=artifacts,
                        K_=K_,
                        x_min_raw=x_min_raw,
                        x_max_raw=x_max_raw,
                        train_interval=TRAIN_INTERVAL,
                        risk_free_rate=r,
                        delta_t=T,
                        k_terms=K_TERMS,
                        grid_points=GRID_POINTS_FOURIER,
                        dask_client=DASK_CLIENT,
                        debug=bool(cfg.get("debug_price_stats", False)),
                        debug_label=debug_label,
                    )
                else:
                    est_price = estimate_price_ibp(
                        weights=weights,
                        artifacts=artifacts,
                        K_=K_,
                        x_min_raw=x_min_raw,
                        x_max_raw=x_max_raw,
                        train_interval=TRAIN_INTERVAL,
                        risk_free_rate=r,
                        delta_t=T,
                        k_terms=K_TERMS,
                        grid_points=GRID_POINTS_FOURIER,
                        dask_client=DASK_CLIENT,
                        debug=bool(cfg.get("debug_price_stats", False)),
                        debug_label=debug_label,
                    )

                run_seconds = float(t1 - t0)
                results.append({
                    "mode": mode_name,
                    "strike": K_,
                    "dataset_size": n_train,
                    "test_points": n_test,
                    "rep": rep + 1,
                    "estimated_price": est_price,
                    "theoretical_price": float(theor_price),
                    "test_mse": test_mse,
                    "seconds": run_seconds,
                    "n_qubits_by_feature": int(cfg["n_qubits_by_feature"]),
                    "n_layers": int(cfg["n_layers"]),
                })

                run_counter += 1
                elapsed = perf_counter() - global_t0
                avg_per_run = elapsed / max(run_counter, 1)
                eta_seconds = (total_runs - run_counter) * avg_per_run

                progress.update(1)
                progress.set_postfix({
                    "mode": mode_name,
                    "K": K_,
                    "n": n_train,
                    "rep": rep + 1,
                    "run_s": f"{run_seconds:.1f}",
                    "eta": format_eta(eta_seconds),
                })

                print(
                    f"mode={mode_name} K={K_} n_train={n_train} rep={rep + 1} "
                    f"price={est_price:.6f} run_s={run_seconds:.2f} ETA={format_eta(eta_seconds)}"
                )

progress.close()

total_elapsed = perf_counter() - global_t0
print(f"\nCompleted {run_counter}/{total_runs} runs in {format_eta(total_elapsed)}")

results_df = pd.DataFrame(results)
results_df.head()


## 5. Summaries

In [ ]:
summary_df = results_df.groupby(["mode", "strike", "dataset_size"], as_index=False).agg(
    mean_price=("estimated_price", "mean"),
    std_price=("estimated_price", "std"),
    mean_test_mse=("test_mse", "mean"),
    mean_seconds=("seconds", "mean"),
)
summary_df


## 6. Price Curves by Dataset Size


In [ ]:
colors = ["#377eb8", "#ff7f00", "#4daf4a"]

for mode_name, cfg in MODE_CONFIGS.items():
    dataset_sizes = cfg["train_sizes"]
    df_mode = results_df[results_df["mode"] == mode_name].copy()

    plt.figure(figsize=(10, 7))

    for K_, color in zip(STRIKES, colors, strict=False):
        df_k = df_mode[df_mode["strike"] == K_]

        means = []
        stds = []
        for n_ in dataset_sizes:
            vals = df_k[df_k["dataset_size"] == n_]["estimated_price"].values
            means.append(np.nanmean(vals) if vals.size > 0 else np.nan)
            stds.append(np.nanstd(vals, ddof=1) if vals.size > 1 else 0.0)

        plt.errorbar(
            dataset_sizes,
            means,
            yerr=stds,
            fmt="-o",
            color=color,
            label=f"K = {K_}",
            capsize=5,
            markersize=8,
            elinewidth=1.5,
        )

        # Theoretical line for each strike
        p_theo = bs_put_price(S0, K_, r, sigma, T)
        plt.axhline(p_theo, color=color, linestyle="--", linewidth=0.9, alpha=0.9)

        # Outliers by IQR
        for n_ in dataset_sizes:
            vals = df_k[df_k["dataset_size"] == n_]["estimated_price"].dropna().values
            if vals.size < 4:
                continue
            q1, q3 = np.percentile(vals, [25, 75])
            iqr = q3 - q1
            low = q1 - 1.5 * iqr
            high = q3 + 1.5 * iqr
            outliers = vals[(vals < low) | (vals > high)]
            if outliers.size > 0:
                plt.scatter(
                    np.full(outliers.shape[0], n_),
                    outliers,
                    marker="x",
                    color=color,
                    s=45,
                    alpha=0.95,
                    label=f"Outliers K={K_}" if n_ == dataset_sizes[0] else None,
                )

    plt.title(f"Estimated prices by dataset size | mode={mode_name} | test_points={cfg['test_points']}")
    plt.xlabel("Size of the dataset (n_train)")
    plt.ylabel("Price")
    plt.xticks(dataset_sizes)
    plt.grid(alpha=0.25, axis="y")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()